In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료!")

In [ ]:
# 데이터 로드
train_df = pd.read_csv('../data/train.csv')
val_df = pd.read_csv('../data/val.csv')
test_df = pd.read_csv('../data/test.csv')

feature_cols = ['total_words', 'filler_count', 'filler_ratio', 
                'vocab_diversity', 'avg_word_len']

X_train = train_df[feature_cols]
y_train = train_df['label']
X_val = val_df[feature_cols]
y_val = val_df['label']
X_test = test_df[feature_cols]
y_test = test_df['label']

print(f"Train: {len(X_train)}개")
print(f"Val:   {len(X_val)}개")
print(f"Test:  {len(X_test)}개")
print(f"\nfeature 목록: {feature_cols}")

In [ ]:
# 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("스케일링 완료!")
print(f"Train 평균: {X_train_scaled.mean(axis=0).round(3)}")
print(f"Train 표준편차: {X_train_scaled.std(axis=0).round(3)}")

In [ ]:
# 모델 1 - Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# Val 성능
lr_val_pred = lr_model.predict(X_val_scaled)
lr_val_acc = accuracy_score(y_val, lr_val_pred)
lr_val_f1 = f1_score(y_val, lr_val_pred)

print("=== Logistic Regression ===")
print(f"Val Accuracy: {lr_val_acc:.4f} ({lr_val_acc*100:.1f}%)")
print(f"Val F1-score: {lr_val_f1:.4f}")
print(f"\n{classification_report(y_val, lr_val_pred)}")

In [ ]:
# filler_ratio 제외한 feature로 재시도
feature_cols2 = ['total_words', 'vocab_diversity', 'avg_word_len']

X_train2 = train_df[feature_cols2]
X_val2 = val_df[feature_cols2]
X_test2 = test_df[feature_cols2]

scaler2 = StandardScaler()
X_train2_scaled = scaler2.fit_transform(X_train2)
X_val2_scaled = scaler2.transform(X_val2)
X_test2_scaled = scaler2.transform(X_test2)

lr_model2 = LogisticRegression(random_state=42, max_iter=1000)
lr_model2.fit(X_train2_scaled, y_train)

lr_val_pred2 = lr_model2.predict(X_val2_scaled)
print("=== Logistic Regression (filler 제외) ===")
print(f"Val Accuracy: {accuracy_score(y_val, lr_val_pred2):.4f}")
print(f"Val F1-score: {f1_score(y_val, lr_val_pred2):.4f}")
print(f"\n{classification_report(y_val, lr_val_pred2)}")

In [ ]:
# 모델 2 - Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train2_scaled, y_train)

rf_val_pred = rf_model.predict(X_val2_scaled)
rf_val_acc = accuracy_score(y_val, rf_val_pred)
rf_val_f1 = f1_score(y_val, rf_val_pred)

print("=== Random Forest ===")
print(f"Val Accuracy: {rf_val_acc:.4f} ({rf_val_acc*100:.1f}%)")
print(f"Val F1-score: {rf_val_f1:.4f}")
print(f"\n{classification_report(y_val, rf_val_pred)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Logistic Regression Confusion Matrix
cm_lr = confusion_matrix(y_val, lr_val_pred2)
sns.heatmap(cm_lr, annot=True, fmt='d', ax=axes[0], cmap='Blues')
axes[0].set_title('Logistic Regression\nConfusion Matrix')
axes[0].set_xlabel('예측')
axes[0].set_ylabel('실제')
axes[0].set_xticklabels(['Poor(0)', 'Good(1)'])
axes[0].set_yticklabels(['Poor(0)', 'Good(1)'])

# Random Forest Confusion Matrix
cm_rf = confusion_matrix(y_val, rf_val_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', ax=axes[1], cmap='Blues')
axes[1].set_title('Random Forest\nConfusion Matrix')
axes[1].set_xlabel('예측')
axes[1].set_ylabel('실제')
axes[1].set_xticklabels(['Poor(0)', 'Good(1)'])
axes[1].set_yticklabels(['Poor(0)', 'Good(1)'])

plt.tight_layout()
plt.savefig('../results/baseline_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("그래프 저장 완료!")

In [ ]:
# Test 최종 성능
lr_test_pred = lr_model2.predict(X_test2_scaled)
rf_test_pred = rf_model.predict(X_test2_scaled)

print("=== 최종 Test 성능 ===")
print(f"{'모델':<25} {'Accuracy':>10} {'F1-score':>10}")
print("-" * 45)
print(f"{'Logistic Regression':<25} {accuracy_score(y_test, lr_test_pred):>10.4f} {f1_score(y_test, lr_test_pred):>10.4f}")
print(f"{'Random Forest':<25} {accuracy_score(y_test, rf_test_pred):>10.4f} {f1_score(y_test, rf_test_pred):>10.4f}")

# Feature Importance (Random Forest)
plt.figure(figsize=(8, 4))
importance = pd.Series(rf_model.feature_importances_, index=feature_cols2)
importance.sort_values().plot(kind='barh', color='steelblue')
plt.title('Random Forest Feature Importance')
plt.xlabel('중요도')
plt.savefig('../results/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("저장 완료!")